In [2]:
import pandas as pd
import numpy as np

# 1. Load data
df_prices = pd.read_csv("data/GOOG_prices.csv", index_col=0)
df_sentiment = pd.read_csv("data/GOOG_sentiment_by_day.csv")

df_prices.index = pd.to_datetime(df_prices.index).date
df_sentiment["date"] = pd.to_datetime(df_sentiment["date"]).dt.date

# 2. Calculate daily return
df_prices["return"] = df_prices["Close"].pct_change() * 100

# 3. Merge
df = pd.merge(df_sentiment, df_prices, left_on="date", right_index=True)

print(f"Combined dataset: {len(df)} days")
print(df[["date", "sentiment_score", "return", "Close"]])

Combined dataset: 9 days
          date  sentiment_score    return       Close
1   2026-02-18              0.0  0.369864  303.726044
2   2026-02-19              0.0 -0.125034  303.346283
3   2026-02-20              0.0  3.735675  314.678314
5   2026-02-23              0.0 -1.019369  311.470581
6   2026-02-24              1.0 -0.247034  310.701141
7   2026-02-25              0.0  0.678623  312.809631
8   2026-03-03             -1.0 -0.913959  303.346283
9   2026-03-05              0.0 -0.837049  300.698151
12  2026-03-09              0.0  2.656920  306.010010


In [3]:
# 4. Next day return (sentiment predicts next day)
df = df.sort_values("date").reset_index(drop=True)
df["next_day_return"] = df["return"].shift(-1)

# Drop last row (no next day return)
df = df.dropna(subset=["next_day_return"])

print(f"Final dataset: {len(df)} days")
print(df[["date", "sentiment_score", "return", "next_day_return"]])

Final dataset: 8 days
         date  sentiment_score    return  next_day_return
0  2026-02-18              0.0  0.369864        -0.125034
1  2026-02-19              0.0 -0.125034         3.735675
2  2026-02-20              0.0  3.735675        -1.019369
3  2026-02-23              0.0 -1.019369        -0.247034
4  2026-02-24              1.0 -0.247034         0.678623
5  2026-02-25              0.0  0.678623        -0.913959
6  2026-03-03             -1.0 -0.913959        -0.837049
7  2026-03-05              0.0 -0.837049         2.656920


In [4]:
# 5. Correlation analysis
correlation = df["sentiment_score"].corr(df["next_day_return"])
print(f"Correlation between sentiment and next day return: {correlation:.4f}")

# Save final dataset
df.to_csv("data/GOOG_final_dataset.csv", index=False)
print("Saved")

Correlation between sentiment and next day return: 0.2275
Saved
